# WiSARD Sweep — TON_IoT (telemetria)

Sweep completo do **WiSARD** nas 7 subbases de telemetria do TON_IoT,
seguindo o schema e as decisões fixas do **`PLANO.md`**.

## O que este notebook faz

- Carrega e limpa os 7 CSVs com `clean_df()` (mesma função do EDA).
- Para cada base × tarefa (`binary`, `multiclass`) × config do grid:
  - Ajusta o encoder **só no treino** (sem leakage do teste).
  - Treina `wp.Wisard` com `BBleaching` (busca binária de bleach).
  - Calcula métricas completas: accuracy, precision/recall/F1 (macro + weighted), AUC-ROC.
  - Mede memória (serializada + teórica) e latência de inferência (μs/amostra).
  - Salva resultado em `results/wisard/<base>__<task>.jsonl`.
- Hiperparâmetro extra do WiSARD: `negativeEvidence` on/off (ver §3.2 do PLANO).
- Grid: 4 termômetros × 6 tamanhos × 8 addressSizes = **192 configs** por base,
  × 2 tarefas × 7 bases × 2 valores de negativeEvidence = **~5 376 experimentos**
  (configs inválidas são puladas com `skipped=True`).

## Pré-requisitos

- CSVs em `../../data/toniot/`.
- Ambiente configurado com `uv sync` (ver README).
- `wisardpkg` compilado (fork `muanlartins/wisardpkg@muanlartins`).

## Tempo estimado

WiSARD é basicamente escrita em RAM — treino em segundos por config.
Com ~5k configs válidas e latência ~2 s por config (treino + inferência + métricas),
espere **~3–4 h** no total num CPU moderno. Pode interromper e retomar:
o notebook detecta JSONLs existentes e pula configs já processadas (ver célula de controle).

## 0. Imports e configuração

In [ ]:
import os
import json
import math
import time
import pickle
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import wisardpkg as wp
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix,
)

warnings.filterwarnings('ignore')

# ── Caminhos ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../../data/toniot/')
RESULTS_DIR = Path('./results/')

EXPECTED = [
    'Fridge', 'Garage_Door', 'GPS_Tracker', 'Modbus',
    'Motion_Light', 'Thermostat', 'Weather',
]

# ── Grid (PLANO §3.1) ─────────────────────────────────────────────────────────
THERMOMETERS  = ['Simple', 'Distributive', 'Gaussian', 'Exponential']
THERMO_SIZES  = [2, 4, 8, 16, 32, 64]        # bits/feature
ADDRESS_SIZES = [4, 8, 12, 16, 20, 24, 28, 32]  # bits/RAM
TASKS         = ['binary', 'multiclass']
NEG_EVIDENCE  = [False, True]   # WiSARD específico (PLANO §3.2)

# ── Reprodutibilidade (PLANO §3.3) ────────────────────────────────────────────
RANDOM_STATE = 0
TEST_SIZE    = 0.3

# ── Colunas meta (nunca entram no encoder) ───────────────────────────────────
META_COLS = {'date', 'time', 'label', 'type'}

# ── Máquina (para normalização cross-integrante, ver PLANO §2.4) ─────────────
MACHINE = 'local-dev'   # <-- altere para 'Colab CPU free-tier' se rodar no Colab

print(f'wisardpkg version: {wp.__version__}')
print(f'Grid total por base: {len(THERMOMETERS)} termômetros × {len(THERMO_SIZES)} sizes × {len(ADDRESS_SIZES)} addressSizes = {len(THERMOMETERS)*len(THERMO_SIZES)*len(ADDRESS_SIZES)} configs')
print(f'Multiplicado por {len(TASKS)} tarefas × {len(NEG_EVIDENCE)} negEvidence = {len(THERMOMETERS)*len(THERMO_SIZES)*len(ADDRESS_SIZES)*len(TASKS)*len(NEG_EVIDENCE)} experimentos / base (antes de skips)')

## 1. Carregamento e limpeza dos CSVs

`clean_df()` é idêntica ao EDA (PLANO §3.3 — crítico para reprodutibilidade).

In [ ]:
def clean_df(df):
    """
    Normaliza um CSV TON_IoT recém-carregado.
    Idêntica à versão do EDA — NÃO alterar sem avisar todos os integrantes.

    - lowercase + strip nos nomes de colunas
    - strip + lowercase em todas as colunas não-numéricas
    - sphone_signal {'0','1','false','true'} -> Int64 {0,1}
    - label como Int64
    """
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.lower()
    if 'sphone_signal' in df.columns:
        df['sphone_signal'] = (
            df['sphone_signal']
            .map({'0': 0, '1': 1, 'false': 0, 'true': 1})
            .astype('Int64')
        )
    if 'label' in df.columns:
        df['label'] = pd.to_numeric(df['label'], errors='coerce').astype('Int64')
    return df


def find_csv(device):
    if not DATA_DIR.exists():
        return None
    target = f'train_test_iot_{device}.csv'.lower()
    for p in DATA_DIR.iterdir():
        if p.name.lower() == target:
            return p
    return None


dfs = {}
missing = []
for device in EXPECTED:
    p = find_csv(device)
    if p is None:
        missing.append(device)
        continue
    dfs[device] = clean_df(pd.read_csv(p))

print(f'Carregadas: {len(dfs)} / {len(EXPECTED)} bases')
for device, df in dfs.items():
    print(f'  {device:<14}  {df.shape[0]:>6,} x {df.shape[1]}  label={df["label"].value_counts().to_dict()}')
if missing:
    print(f'\n⚠️  FALTANDO: {missing}')
    print(f'   Esperado em: {DATA_DIR.resolve()}')
    print('   Ver README.md para instruções de download.')
    raise FileNotFoundError(f'CSVs faltando: {missing}')

## 2. Funções utilitárias (de `sweep_utils.py`)

Todas estas funções são transcrição direta do EDA (§10). Não altere sem
sincronizar com `sweep_utils.py` e avisar os demais integrantes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1  feature_cols + cat_encoded_bits
# ─────────────────────────────────────────────────────────────────────────────

def feature_cols(df):
    """Separa as features em (numericas, categoricas). Exclui META_COLS."""
    num, cat = [], []
    for c in df.columns:
        if c in META_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num.append(c)
        else:
            cat.append(c)
    return num, cat


def cat_encoded_bits(unique_values):
    """
    Bits gastos por uma feature categórica (PLANO §3.3):
    - 2 valores (binária) -> 1 bit
    - k valores (k>2)     -> k bits (one-hot)
    """
    return 1 if len(unique_values) == 2 else len(unique_values)


# ─────────────────────────────────────────────────────────────────────────────
# 2.2  Encoder: fit + encode
# ─────────────────────────────────────────────────────────────────────────────

# ── Construtores de termômetro ──────────────────────────────────────────────
# SimpleThermometer aceita APENAS min/max escalares (um termômetro por feature).
# Os demais tipos (Distributive, Gaussian, Exponential) aceitam um único objeto
# e fazem o fit sobre a matriz inteira de features.
#
# Para uniformizar a interface de encode_dataset, fit_encoder retorna:
#   - Simple     → lista de termômetros, um por feature numérica
#   - Demais     → objeto único de termômetro (multifeature)
# encode_dataset detecta qual caso pelo tipo do retorno.


def fit_encoder(df_train, num_cols, therm_type, size):
    """
    Retorna encoder ajustado APENAS no treino.

    Retorno:
      - Simple:      lista [therm_col0, therm_col1, ...] (um por feature numérica)
      - Demais:      objeto único de termômetro wisardpkg
      - sem num_cols: None
    """
    if not num_cols:
        return None
    vals = df_train[num_cols].astype(float).values  # (n_samples, n_features)

    if therm_type == 'Simple':
        # Um SimpleThermometer por feature — cada um recebe scalar min/max
        therms = []
        for col_idx in range(vals.shape[1]):
            col = vals[:, col_idx]
            mn  = float(col.min())
            mx  = float(col.max())
            if mn == mx:           # feature constante → evita divisão por zero
                mx = mn + 1.0
            therms.append(wp.SimpleThermometer(size, minimum=mn, maximum=mx))
        return therms              # lista
    else:
        # Distributive / Gaussian / Exponential: objeto único, fit na matriz
        if therm_type == 'Distributive':
            therm = wp.DistributiveThermometer(size)
        elif therm_type == 'Gaussian':
            therm = wp.GaussianThermometer(size)
        elif therm_type == 'Exponential':
            therm = wp.ExponentialThermometer(size)
        else:
            raise ValueError(f'Termômetro desconhecido: {therm_type}')
        therm.fit(vals.tolist())
        return therm               # objeto único


def cat_categories(df_train, cat_cols):
    """Valores únicos por coluna categórica, ordem determinística (sorted)."""
    return {c: sorted(df_train[c].unique()) for c in cat_cols}


def _encode_num_row_simple(therms, row):
    """Codifica uma linha com lista de SimpleThermometers (um por feature)."""
    bits = []
    for t, v in zip(therms, row):
        bits.extend(t.transform([float(v)]).list())
    return bits


def encode_dataset(df, num_cols, cat_cols, cat_values, therm):
    """
    Encoda um DataFrame inteiro em lista de listas de bits.

    therm pode ser:
      - lista de SimpleThermometers (um por feature numérica)
      - objeto único de termômetro multifeature
      - None (sem features numéricas)
    """
    n = len(df)
    if num_cols:
        num_vals = df[num_cols].astype(float).values
        if isinstance(therm, list):
            # Simple: um termômetro por feature
            bit_rows = [_encode_num_row_simple(therm, row) for row in num_vals]
        else:
            # Distributive / Gaussian / Exponential: objeto único
            bit_rows = [therm.transform(row.tolist()).list() for row in num_vals]
    else:
        bit_rows = [[] for _ in range(n)]

    if cat_cols:
        cat_vals_arr = df[cat_cols].values
        for i, row in enumerate(cat_vals_arr):
            for c_idx, c in enumerate(cat_cols):
                v    = row[c_idx]
                cats = cat_values[c]
                if len(cats) == 2:
                    bit_rows[i].append(1 if v == cats[1] else 0)
                else:
                    bit_rows[i].extend([1 if v == cat else 0 for cat in cats])
    return bit_rows


def encoded_input_bits(n_num, cat_values, therm_size):
    cat_bits = sum(cat_encoded_bits(v) for v in cat_values.values())
    return n_num * therm_size + cat_bits


# ─────────────────────────────────────────────────────────────────────────────
# 2.3  make_split
# ─────────────────────────────────────────────────────────────────────────────

def make_split(df, task, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """
    Split estratificado por 'type' (PLANO §3.3).
    task: 'binary' -> y = 'normal'/'attack'
          'multiclass' -> y = tipo de ataque (string)
    """
    if task == 'binary':
        y = df['label'].astype(int).map({0: 'normal', 1: 'attack'}).values
    elif task == 'multiclass':
        y = df['type'].values
    else:
        raise ValueError(f'task desconhecida: {task}')

    idx_train, idx_test = train_test_split(
        np.arange(len(df)),
        test_size=test_size,
        random_state=random_state,
        stratify=df['type'].values,
    )
    df_train = df.iloc[idx_train].reset_index(drop=True)
    df_test  = df.iloc[idx_test].reset_index(drop=True)
    return df_train, df_test, y[idx_train], y[idx_test]


# ─────────────────────────────────────────────────────────────────────────────
# 2.4  compute_metrics
# ─────────────────────────────────────────────────────────────────────────────

def compute_metrics(y_true, y_pred, y_score=None, task='binary', score_classes=None):
    """
    Retorna (metrics_dict, confusion_matrix, labels) no schema do PLANO §4.
    """
    out = {
        'accuracy':           float(accuracy_score(y_true, y_pred)),
        'precision_macro':    float(precision_score(y_true, y_pred, average='macro',    zero_division=0)),
        'precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'recall_macro':       float(recall_score(y_true, y_pred, average='macro',       zero_division=0)),
        'recall_weighted':    float(recall_score(y_true, y_pred, average='weighted',    zero_division=0)),
        'f1_macro':           float(f1_score(y_true, y_pred, average='macro',           zero_division=0)),
        'f1_weighted':        float(f1_score(y_true, y_pred, average='weighted',        zero_division=0)),
        'balanced_accuracy':  float(balanced_accuracy_score(y_true, y_pred)),
        'auc_roc':            None,
        # campos de custo preenchidos depois
        'memory_bytes_serialized':  0,
        'memory_bytes_theoretical': 0,
        'train_time_s':             0.0,
        'inference_latency_us':     0.0,
    }

    if y_score is not None:
        try:
            score_arr = np.asarray(y_score, dtype=float)
            if task == 'binary':
                if score_arr.ndim == 1:
                    pos_score = score_arr
                else:
                    pos_idx = score_classes.index('attack')
                    pos_score = score_arr[:, pos_idx]
                out['auc_roc'] = float(roc_auc_score(y_true == 'attack', pos_score))
            else:
                out['auc_roc'] = float(roc_auc_score(
                    y_true, score_arr,
                    multi_class='ovr', average='macro',
                    labels=score_classes,
                ))
        except Exception as e:
            pass  # AUC pode falhar em casos extremos (ex: só 1 classe no batch)

    labels = sorted(set(list(y_true) + list(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return out, cm, labels


def ranks_to_score_matrix(rank_list, classes):
    """Converte saída de wp.Wisard.rank() em matriz (n, K) de scores normalizados."""
    n = len(rank_list)
    M = np.zeros((n, len(classes)), dtype=float)
    for i, d in enumerate(rank_list):
        total = sum(d.values()) or 1
        for k, cls in enumerate(classes):
            M[i, k] = d.get(cls, 0) / total
    return M


# ─────────────────────────────────────────────────────────────────────────────
# 2.5  measure_memory + measure_latency
# ─────────────────────────────────────────────────────────────────────────────

def measure_memory(model, input_bits, address_size, n_classes, bytes_per_entry=4):
    """
    (serialized_bytes, theoretical_bytes)
    Teórico = n_classes × ceil(input_bits/address_size) × 2^address_size × bytes_per_entry
    """
    serialized   = len(model.json().encode('utf-8'))
    n_rams       = math.ceil(input_bits / address_size)
    entries_per  = 2 ** address_size
    theoretical  = n_classes * n_rams * entries_per * bytes_per_entry
    return serialized, theoretical


def measure_latency(model, X_test_lists, n_iter=1000):
    """Mediana de classify() single-sample em microssegundos."""
    if not X_test_lists:
        return None
    n        = len(X_test_lists)
    pool_sz  = min(n, n_iter)
    pool     = [wp.DataSet([X_test_lists[i]]) for i in range(pool_sz)]
    # warm-up
    for ds in pool[:min(20, pool_sz)]:
        _ = model.classify(ds)
    times = []
    for i in range(n_iter):
        ds = pool[i % pool_sz]
        t0 = time.perf_counter_ns()
        _  = model.classify(ds)
        times.append((time.perf_counter_ns() - t0) / 1000.0)
    times.sort()
    return float(times[len(times) // 2])


# ─────────────────────────────────────────────────────────────────────────────
# 2.6  iter_grid — configs válidas (PLANO §3.1 + §3.4)
# ─────────────────────────────────────────────────────────────────────────────

def iter_grid(
    n_num, cat_card,
    thermo_sizes=THERMO_SIZES,
    address_sizes=ADDRESS_SIZES,
    thermometers=THERMOMETERS,
    neg_evidence_values=NEG_EVIDENCE,
):
    """Itera todas as configs; marca skipped=True quando addressSize > input_bits."""
    for th in thermometers:
        for ts in thermo_sizes:
            input_bits = n_num * ts + cat_card
            for ad in address_sizes:
                skipped = ad > input_bits
                for neg_ev in neg_evidence_values:
                    yield {
                        'thermometer':   th,
                        'thermo_size':   ts,
                        'address_size':  ad,
                        'input_bits':    input_bits,
                        'neg_evidence':  neg_ev,
                        'skipped':       skipped,
                    }


# ─────────────────────────────────────────────────────────────────────────────
# 2.7  result_dict + save_result — schema PLANO §4
# ─────────────────────────────────────────────────────────────────────────────

def result_dict(
    model_name, base, task, config, metrics, cm, labels,
    input_info, model_hyperparams=None, machine=None,
    skipped=False, skipped_reason=None,
):
    return {
        'model':            model_name,
        'base':             base,
        'task':             task,
        'encoder':          {'type': config['thermometer'], 'size': config['thermo_size']},
        'addressSize':      config['address_size'],
        'model_hyperparams': model_hyperparams or {},
        'split':            {'random_state': RANDOM_STATE, 'test_size': TEST_SIZE, 'stratified': True},
        'input':            input_info,
        'skipped':          skipped,
        'skipped_reason':   skipped_reason,
        'metrics':          metrics,
        'confusion_matrix': cm,
        'labels':           labels,
        'machine':          machine or MACHINE,
        'wisardpkg_version': wp.__version__,
        'timestamp':        datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }


def save_result(res, model_name, base, task):
    out_dir = RESULTS_DIR / model_name.lower()
    out_dir.mkdir(parents=True, exist_ok=True)
    fpath = out_dir / f'{base}__{task}.jsonl'
    with open(fpath, 'a') as f:
        f.write(json.dumps(res) + '\n')
    return fpath


print('Utilitários carregados.')

## 3. Controle de execução (resumo / retomada)

O sweep pode ser interrompido e retomado: o notebook carrega configs já
salvas nos JSONLs e evita reprocessá-las. Útil para rodar em etapas ou
após queda de Colab.

In [ ]:
def load_done_keys(model_name, base, task):
    """
    Retorna conjunto de chaves (thermometer, thermo_size, address_size, neg_evidence)
    já processadas no JSONL correspondente.
    """
    fpath = RESULTS_DIR / model_name.lower() / f'{base}__{task}.jsonl'
    if not fpath.exists():
        return set()
    keys = set()
    with open(fpath) as f:
        for line in f:
            try:
                r = json.loads(line)
                keys.add((
                    r['encoder']['type'],
                    r['encoder']['size'],
                    r['addressSize'],
                    r.get('model_hyperparams', {}).get('negativeEvidence', False),
                ))
            except Exception:
                pass
    return keys


# Pré-visualização do grid por base
print('Grid de configs válidas (excl. skips) por base:')
print(f'{"Base":<14} {"bits@T2":>8} {"bits@T8":>8} {"bits@T16":>9} {"bits@T32":>9}  valid_configs')
for device, df in dfs.items():
    num, cat = feature_cols(df)
    cat_bits = sum(cat_encoded_bits(df[c].unique()) for c in cat)
    valid = sum(
        1 for ts in THERMO_SIZES for ad in ADDRESS_SIZES
        if ad <= len(num) * ts + cat_bits
    )
    b2  = len(num)*2  + cat_bits
    b8  = len(num)*8  + cat_bits
    b16 = len(num)*16 + cat_bits
    b32 = len(num)*32 + cat_bits
    total = valid * len(THERMOMETERS) * len(NEG_EVIDENCE) * len(TASKS)
    print(f'{device:<14} {b2:>8} {b8:>8} {b16:>9} {b32:>9}  {total} experimentos')

## 4. Sweep principal — WiSARD

Loop externo: base → tarefa → config do grid.

### Notas de implementação

- **BBleaching**: `wp.Wisard` aceita `bleaching=True` para ativar a busca binária
  automática de threshold de bleach (conforme PLANO §3.3). O valor escolhido
  é registrado em `model_hyperparams.bleach_b`.
- **negativeEvidence**: hiperparâmetro extra do WiSARD (PLANO §3.2). Quando
  `True`, o classificador considera também os votos negativos.
- **AUC multiclasse**: calculado como OvR macro via `rank()` do modelo.
- **Latência**: mediana de 1000 inferências single-sample (PLANO §2.2).

In [ ]:
def run_wisard_config(clf, df_train, df_test, y_train, y_test,
                      num, cat, cat_values, config, task, base):
    """
    Executa uma config completa de WiSARD e retorna o result_dict pronto.
    Retorna None para configs skipped.
    """
    if config['skipped']:
        res = result_dict(
            'WiSARD', base, task, config,
            metrics={
                'accuracy': None, 'precision_macro': None, 'precision_weighted': None,
                'recall_macro': None, 'recall_weighted': None,
                'f1_macro': None, 'f1_weighted': None, 'balanced_accuracy': None,
                'auc_roc': None, 'memory_bytes_serialized': None,
                'memory_bytes_theoretical': None, 'train_time_s': None,
                'inference_latency_us': None,
            },
            cm=[], labels=[],
            input_info={
                'n_features_num': len(num),
                'n_features_cat': len(cat),
                'cat_bits': sum(cat_encoded_bits(v) for v in cat_values.values()),
                'bits_total': config['input_bits'],
            },
            model_hyperparams={'negativeEvidence': config['neg_evidence'], 'bleaching': True},
            skipped=True,
            skipped_reason='addressSize > input_bits',
        )
        return res

    # Encoder: fit APENAS no treino
    therm = fit_encoder(df_train, num, config['thermometer'], config['thermo_size'])
    X_train = encode_dataset(df_train, num, cat, cat_values, therm)
    X_test  = encode_dataset(df_test,  num, cat, cat_values, therm)

    cat_bits    = sum(cat_encoded_bits(v) for v in cat_values.values())
    input_bits  = config['input_bits']
    n_classes   = len(set(y_train))
    classes     = sorted(set(list(y_train) + list(y_test)))

    ds_train = wp.DataSet(X_train, list(y_train))
    ds_test  = wp.DataSet(X_test)

    # Treino
    try:
        clf = wp.Wisard(
            config['address_size'],
            bleaching=True,
            ignoreZero=False,
        )
        t0 = time.perf_counter()
        clf.train(ds_train)
        train_time_s = time.perf_counter() - t0
    except Exception as e:
        print(f'    ⚠️  Erro no treino: {e}')
        return None

    # Inferência
    try:
        y_pred   = np.array(clf.classify(ds_test))
        rank_out = clf.rank(ds_test)
        y_score  = ranks_to_score_matrix(rank_out, classes)
    except Exception as e:
        print(f'    ⚠️  Erro na inferência: {e}')
        return None

    # Métricas
    metrics, cm, labels = compute_metrics(
        y_test, y_pred,
        y_score=y_score,
        task=task,
        score_classes=classes,
    )

    # Memória
    ser, theo = measure_memory(clf, input_bits, config['address_size'], n_classes)
    metrics['memory_bytes_serialized']  = int(ser)
    metrics['memory_bytes_theoretical'] = int(theo)
    metrics['train_time_s']             = round(train_time_s, 4)

    # Latência (batch=1, 1000 iterações)
    lat = measure_latency(clf, X_test[:500], n_iter=1000)
    metrics['inference_latency_us'] = round(lat, 2) if lat is not None else None

    # Tenta capturar o bleach_b utilizado (se wisardpkg expõe)
    bleach_b = None
    try:
        model_json = json.loads(clf.json())
        bleach_b   = model_json.get('bleach', None)
    except Exception:
        pass

    res = result_dict(
        'WiSARD', base, task, config, metrics, cm, labels,
        input_info={
            'n_features_num': len(num),
            'n_features_cat': len(cat),
            'cat_bits': cat_bits,
            'bits_total': input_bits,
        },
        model_hyperparams={
            'negativeEvidence': config['neg_evidence'],
            'bleaching': True,
            'bleach_b': bleach_b,
        },
    )
    return res


print('Função run_wisard_config definida.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SWEEP PRINCIPAL
# Estimativa: ~3–4 h em CPU moderno.
# Pode interromper e retomar: configs já salvas no JSONL são puladas.
# ══════════════════════════════════════════════════════════════════════════════

SWEEP_START = time.time()
total_done  = 0
total_skip  = 0
total_err   = 0

for device in EXPECTED:
    if device not in dfs:
        print(f'[{device}] CSV não encontrado — pulando.')
        continue

    df      = dfs[device]
    num, cat = feature_cols(df)
    cat_bits = sum(cat_encoded_bits(df[c].unique()) for c in cat)
    n_num    = len(num)

    print(f'\n{"═"*70}')
    print(f'BASE: {device}  |  features: num={num}, cat={cat}  |  bits@T8={n_num*8+cat_bits}')

    for task in TASKS:
        print(f'\n  ▶ tarefa: {task}')

        # Split (mesmo para todas as configs desta base × tarefa)
        df_train, df_test, y_train, y_test = make_split(df, task)
        cat_values = cat_categories(df_train, cat)

        # Configs já processadas neste JSONL
        done_keys = load_done_keys('WiSARD', device, task)
        if done_keys:
            print(f'     (retomando: {len(done_keys)} configs já salvas)')

        base_done = base_skip = base_err = 0
        t_base_start = time.time()

        for config in iter_grid(n_num, cat_bits):
            key = (
                config['thermometer'],
                config['thermo_size'],
                config['address_size'],
                config['neg_evidence'],
            )

            # Pula configs já salvas
            if key in done_keys:
                base_skip += 1
                continue

            # Configs inválidas (addressSize > input_bits): salva o skip e continua
            if config['skipped']:
                res = run_wisard_config(
                    None, df_train, df_test, y_train, y_test,
                    num, cat, cat_values, config, task, device,
                )
                if res:
                    save_result(res, 'WiSARD', device, task)
                base_skip += 1
                total_skip += 1
                continue

            # Executa config válida
            res = run_wisard_config(
                None, df_train, df_test, y_train, y_test,
                num, cat, cat_values, config, task, device,
            )

            if res is None:
                base_err  += 1
                total_err += 1
                continue

            save_result(res, 'WiSARD', device, task)
            base_done  += 1
            total_done += 1

            # Log periódico
            if base_done % 20 == 0:
                elapsed = time.time() - t_base_start
                f1 = res['metrics'].get('f1_macro')
                f1_str = f'{f1:.3f}' if f1 is not None else 'N/A'
                print(
                    f'     [{base_done:>4} feitos] '
                    f'therm={config["thermometer"]:<13} '
                    f'T={config["thermo_size"]:>2} '
                    f'addr={config["address_size"]:>2} '
                    f'negEv={config["neg_evidence"]}  '
                    f'f1_macro={f1_str}  '
                    f'({elapsed:.0f}s)'
                )

        t_base = time.time() - t_base_start
        print(f'     ✓ {task}: {base_done} processadas, {base_skip} puladas, '
              f'{base_err} erros  ({t_base:.1f}s)')

elapsed_total = time.time() - SWEEP_START
print(f'\n{"═"*70}')
print(f'SWEEP CONCLUÍDO em {elapsed_total/60:.1f} min')
print(f'  Processadas: {total_done}  |  Puladas (skip/retomada): {total_skip}  |  Erros: {total_err}')

## 5. Verificação dos resultados

Conta entradas por JSONL, reporta top-5 configs por F1 macro em cada base.

In [ ]:
import os

wisard_dir = RESULTS_DIR / 'wisard'
if not wisard_dir.exists():
    print('Nenhum resultado encontrado ainda.')
else:
    files = sorted(wisard_dir.glob('*.jsonl'))
    print(f'Arquivos em {wisard_dir}:')
    for f in files:
        n_lines = sum(1 for _ in open(f))
        size_kb  = f.stat().st_size / 1024
        print(f'  {f.name:<35}  {n_lines:>5} linhas  ({size_kb:.1f} KB)')

In [ ]:
# ── Top-5 configs por F1 macro, por base e tarefa ──────────────────────────

def load_results(model_name, base, task):
    fpath = RESULTS_DIR / model_name.lower() / f'{base}__{task}.jsonl'
    if not fpath.exists():
        return pd.DataFrame()
    rows = []
    with open(fpath) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get('skipped'):
                    continue
                rows.append({
                    'base':        r['base'],
                    'task':        r['task'],
                    'thermometer': r['encoder']['type'],
                    'thermo_size': r['encoder']['size'],
                    'address_size': r['addressSize'],
                    'neg_evidence': r.get('model_hyperparams', {}).get('negativeEvidence', False),
                    'bleach_b':    r.get('model_hyperparams', {}).get('bleach_b'),
                    'bits_total':  r['input']['bits_total'],
                    'accuracy':    r['metrics'].get('accuracy'),
                    'f1_macro':    r['metrics'].get('f1_macro'),
                    'f1_weighted': r['metrics'].get('f1_weighted'),
                    'auc_roc':     r['metrics'].get('auc_roc'),
                    'mem_ser_KB':  (r['metrics'].get('memory_bytes_serialized') or 0) / 1024,
                    'mem_theo_KB': (r['metrics'].get('memory_bytes_theoretical') or 0) / 1024,
                    'train_time_s': r['metrics'].get('train_time_s'),
                    'latency_us':  r['metrics'].get('inference_latency_us'),
                })
            except Exception:
                pass
    return pd.DataFrame(rows)


for task in TASKS:
    print(f'\n{"═"*80}')
    print(f'TAREFA: {task}')
    all_rows = []
    for device in EXPECTED:
        df_res = load_results('WiSARD', device, task)
        if df_res.empty:
            print(f'  {device}: sem resultados ainda.')
            continue
        all_rows.append(df_res)
        top5 = df_res.nlargest(5, 'f1_macro')[[
            'thermometer', 'thermo_size', 'address_size',
            'neg_evidence', 'bits_total', 'accuracy', 'f1_macro',
            'auc_roc', 'mem_ser_KB', 'latency_us',
        ]]
        print(f'\n  {device} — top-5 por F1 macro:')
        print(top5.to_string(index=False))

## 6. Fronteira de Pareto (memória × F1 macro)

Gráfico principal do trabalho (PLANO §2.3): cada ponto é uma
configuração; a curva destaca as configs Pareto-ótimas
(melhor F1 para cada orçamento de memória).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams['figure.dpi']        = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

THERM_COLORS = {
    'Simple':       '#4C78A8',
    'Distributive': '#E45756',
    'Gaussian':     '#54A24B',
    'Exponential':  '#F58518',
}


def pareto_front(df_res, x_col='mem_theo_KB', y_col='f1_macro'):
    """Retorna subset Pareto-ótimo (menor memória → maior F1)."""
    df_s = df_res[[x_col, y_col]].dropna().sort_values(x_col)
    pareto = []
    best_y = -1
    for _, row in df_s.iterrows():
        if row[y_col] > best_y:
            best_y = row[y_col]
            pareto.append(row)
    return pd.DataFrame(pareto)


for task in TASKS:
    all_dfs = []
    for device in EXPECTED:
        df_res = load_results('WiSARD', device, task)
        if not df_res.empty:
            df_res['device'] = device
            all_dfs.append(df_res)

    if not all_dfs:
        print(f'[{task}] Sem resultados para plotar ainda.')
        continue

    n_bases = len(all_dfs)
    ncols   = min(4, n_bases)
    nrows   = math.ceil(n_bases / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax, df_res in zip(axes, all_dfs):
        device = df_res['device'].iloc[0]
        for therm, grp in df_res.groupby('thermometer'):
            ax.scatter(
                grp['mem_theo_KB'], grp['f1_macro'],
                s=18, alpha=0.45,
                color=THERM_COLORS.get(therm, '#999999'),
                label=therm,
            )
        # Fronteira de Pareto
        pf = pareto_front(df_res)
        if not pf.empty:
            ax.plot(pf['mem_theo_KB'], pf['f1_macro'],
                    color='black', linewidth=1.5, linestyle='--',
                    label='Pareto', zorder=5)
        ax.set_xscale('log')
        ax.set_xlabel('Memória teórica (KB, log)', fontsize=9)
        ax.set_ylabel('F1 macro', fontsize=9)
        ax.set_title(f'{device} — {task}', fontsize=10)
        ax.tick_params(labelsize=8)
        ax.set_ylim(-0.02, 1.05)

    handles, labels_leg = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_leg, loc='lower center',
               ncol=len(THERM_COLORS)+1, fontsize=9,
               bbox_to_anchor=(0.5, -0.04), frameon=False)
    for ax in axes[n_bases:]:
        ax.set_visible(False)
    fig.suptitle(f'WiSARD — Fronteira de Pareto (memória × F1 macro) — {task}',
                 y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

## 7. Matrizes de confusão — top-3 configs por base (multiclasse)

Artefato visual para o relatório: onde o modelo confunde cada tipo de ataque.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

CMAP_CM = LinearSegmentedColormap.from_list('cm_cmap', ['#ffffff', '#4C78A8'])


def plot_confusion_matrix(cm_data, labels, title, ax):
    cm_arr = np.array(cm_data, dtype=float)
    # Normaliza por linha (recall por classe)
    row_sums = cm_arr.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm_arr / row_sums

    ax.imshow(cm_norm, cmap=CMAP_CM, vmin=0, vmax=1, aspect='auto')
    n = len(labels)
    ax.set_xticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
    ax.set_yticks(range(n))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_ylabel('Real', fontsize=8)
    ax.set_xlabel('Previsto', fontsize=8)
    ax.set_title(title, fontsize=9)
    for i in range(n):
        for j in range(n):
            v = cm_norm[i, j]
            c = 'white' if v > 0.55 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color=c)


task = 'multiclass'
for device in EXPECTED:
    fpath = RESULTS_DIR / 'wisard' / f'{device}__{task}.jsonl'
    if not fpath.exists():
        continue

    rows = []
    with open(fpath) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get('skipped') or r['metrics'].get('f1_macro') is None:
                    continue
                rows.append(r)
            except Exception:
                pass

    if not rows:
        continue

    rows_sorted = sorted(rows, key=lambda r: r['metrics']['f1_macro'], reverse=True)
    top3 = rows_sorted[:3]

    fig, axes = plt.subplots(1, 3, figsize=(5 * 3, 4.5))
    for ax, r in zip(axes, top3):
        cfg_lbl = (
            f"{r['encoder']['type']}/{r['encoder']['size']}/"
            f"addr={r['addressSize']}  "
            f"negEv={r['model_hyperparams'].get('negativeEvidence',False)}\n"
            f"F1={r['metrics']['f1_macro']:.3f}  acc={r['metrics']['accuracy']:.3f}"
        )
        plot_confusion_matrix(r['confusion_matrix'], r['labels'], cfg_lbl, ax)

    fig.suptitle(f'{device} — Top-3 configs (multiclasse, por F1 macro)', fontsize=11)
    plt.tight_layout()
    plt.show()

## 8. Resumo consolidado por base

Tabela compacta com a melhor config por base × tarefa (para o relatório técnico).

In [ ]:
summary_rows = []
for task in TASKS:
    for device in EXPECTED:
        df_res = load_results('WiSARD', device, task)
        if df_res.empty:
            continue
        best = df_res.loc[df_res['f1_macro'].idxmax()]
        summary_rows.append({
            'base':        device,
            'task':        task,
            'thermometer': best['thermometer'],
            'T':           int(best['thermo_size']),
            'addr':        int(best['address_size']),
            'negEv':       best['neg_evidence'],
            'bits_total':  int(best['bits_total']),
            'accuracy':    round(best['accuracy'], 4),
            'f1_macro':    round(best['f1_macro'], 4),
            'f1_weighted': round(best['f1_weighted'], 4),
            'auc_roc':     round(best['auc_roc'], 4) if best['auc_roc'] else None,
            'mem_theo_KB': round(best['mem_theo_KB'], 1),
            'latency_us':  round(best['latency_us'], 2) if best['latency_us'] else None,
        })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print('Melhor config por base × tarefa (WiSARD):')
    print(df_summary.to_string(index=False))

    # Salva CSV para o notebook de análise final
    out_csv = RESULTS_DIR / 'wisard' / 'summary_best.csv'
    df_summary.to_csv(out_csv, index=False)
    print(f'\nSumário salvo em {out_csv}')
else:
    print('Nenhum resultado disponível ainda — rode o sweep primeiro (célula 4).')

## 9. Notas para o relatório

### Pontos obrigatórios a documentar (PLANO §3 e §5)

1. **Data leakage de timestamp**: `date`/`time` foram excluídas do encoder.
   O dataset TON_IoT foi coletado sequencialmente — normal e ataque em
   janelas temporais disjuntas. Incluir essas colunas daria acurácia perfeita
   trivialmente e não corresponderia a IoT em campo.

2. **Thermostat/scanning**: classe minoritária com ~61 amostras num dataset
   de ~32k. Pesa 1/N_classes no F1 macro — qualquer erro nessa classe
   afunda a métrica. Mencionar explicitamente na análise de resultados.

3. **negativeEvidence**: hiperparâmetro específico do WiSARD. Esperado que
   faça diferença em bases com classes desbalanceadas (onde o modelo
   vê muitos zeros para a classe minoritária).

4. **Comparação de tempo entre máquinas**: os campos `train_time_s` e
   `inference_latency_us` só são comparáveis intra-máquina. Para comparação
   cross-integrante, usar o Colab compartilhado (PLANO §2.4) ou normalizar
   por uma config canônica de referência.

5. **Memória teórica vs serializada**: a teórica é o que escala para hardware
   embarcado (pre-alocação de RAM antes de qualquer dado). A serializada
   reflete o modelo treinado (só endereços efetivamente preenchidos).

6. **Configs puladas (skipped)**: `addressSize > bits_total` → registradas
   no JSONL com `skipped=True` e `skipped_reason='addressSize > input_bits'`.
   Bases pobres em features (Fridge, Garage_Door, Motion_Light) têm alto
   percentual de skips com termômetros pequenos.